In [ ]:
!pip install tsai 
import sys

# put the repo first on sys.path so this geoxts is the one imported
if '/Users/fokia/CRP_SLB' not in sys.path:
    sys.path.insert(0, '/Users/fokia/CRP_SLB')

from geoxts.benchmark_data import *


In [ ]:
import pandas as pd 
!pip install fastparquet
# GR downsampled to one value per foot
df_log = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/Training/logs.parquet', engine='fastparquet')
df_log.loc[df_log['GR'] < -1, 'GR' ] = -1
df_loc = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/Training/loc.parquet', engine='fastparquet')
df_loc = df_loc.reset_index()
df_log = df_loc.merge(df_log, how = 'inner', left_on = 'wellName', right_on = 'wellName')
print(df_log.head())

In [ ]:
df_tops = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/Training/tops.parquet', engine='fastparquet')
df_tops.set_index('well_name', inplace = True)
cols = ['NIOBARA','CODELL', 'FORT_HAYS']
df_tops = df_tops[cols]
df_tops.dropna(inplace = True)
df_tops.head() 

In [ ]:
import numpy as np
# training wells, the ones with a good signature cluster
well_array = np.load('/Users/fokia/CRP_SLB/datasets/colorado/data/data/Training/training_well_list.npy', allow_pickle=True)

In [ ]:
from geoxts.benchmark_data import *

In [ ]:
input_variable = ['GR', 'Depth','Latitude','Longitude']
# training data
X_train, y_train = extract_dataset_Xy(df_log,df_tops,well_array, input_variable, wsize=201, top_list_bool=1 )
X_train.shape, y_train.shape

In [ ]:
df_valid_log

In [ ]:
# validation set for the recall calculation
df_valid_log = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/test/logs_50.parquet', engine='fastparquet')
df_valid_log.loc[df_valid_log['GR'] < -1, 'GR' ] = -1
df_valid_log.loc[df_valid_log['GR'] > 400, 'GR' ] = 400
df_valid_loc = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/test/loc_50.parquet', engine='fastparquet')
df_valid_loc = df_valid_loc.reset_index()
df_valid_log = df_valid_loc.merge(df_valid_log, how = 'inner', left_on = 'wellName', right_on = 'wellName')
df_valid_tops = pd.read_csv('/Users/fokia/CRP_SLB/datasets/colorado/data/data/test/tops_50.csv')
df_valid_tops = df_valid_tops.set_index('wellName')
cols = ['NIOBARA','CODELL', 'FORT_HAYS']
df_valid_tops = df_valid_tops[cols]

In [ ]:
X_valid, y_valid = extract_dataset_Xy(df_valid_log,df_valid_tops,[],input_variable, wsize=201, top_list_bool=0)
X_valid.shape, y_valid.shape

In [ ]:
from tsai.all import *


In [ ]:
X, y, splits = combine_split_data([X_train, X_valid], [y_train, y_valid])
tfms  = [None, [Categorize()]]
dsets = TSDatasets(X, y, tfms=tfms, splits=splits, inplace=True)
dls = TSDataLoaders.from_dsets(dsets.train, dsets.valid, bs=25, batch_tfms=[TSNormalize(by_var=True, range = (0,1))], num_workers=0) 

In [ ]:
dsets

In [ ]:
valid_dl = dls.valid
dls.show_batch()

In [ ]:
# test set for the recall calculation
df_test_log = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/test/logs_100.parquet', engine='fastparquet')
df_test_log.loc[df_test_log['GR'] < -1, 'GR' ] = -1
df_test_log.loc[df_test_log['GR'] > 400, 'GR' ] = 400
df_test_loc = pd.read_parquet('/Users/fokia/CRP_SLB/datasets/colorado/data/data/test/loc_100.parquet', engine='fastparquet')
df_test_loc = df_test_loc.reset_index()
df_test_log = df_test_loc.merge(df_test_log, how = 'inner', left_on = 'wellName', right_on = 'wellName')
df_test_tops = pd.read_csv('/Users/fokia/CRP_SLB/datasets/colorado/data/data/test/tops_100.csv')
df_test_tops = df_test_tops.set_index('wellName')
cols = ['NIOBARA','CODELL', 'FORT_HAYS']
df_test_tops = df_test_tops[cols]

In [ ]:
batch_size = 64
dls = TSDataLoaders.from_dsets(dsets.train, dsets.valid, bs=batch_size)


In [ ]:
import sys
sys.path.append('/Users/fokia/CRP_SLB/datasets/colorado/Transformer')

from PatchTST_backbone import PatchTST_backbone 


In [ ]:
model = PatchTST_backbone(
    c_in=X.shape[1],
    target_window=1,
    context_window=X.shape[2],
    patch_len=16,
    stride=8,
    n_markers=4
)


In [ ]:
%reload_ext autoreload
%autoreload 2 


In [ ]:
df_log

In [ ]:

learn = Learner(dls, model, metrics=accuracy)

# 10 epochs, max lr 1e-3
learn.fit_one_cycle(10, lr_max=1e-3)

# loss curves:
# learn.recorder.plot_loss()


In [ ]:
learn.recorder.plot_loss()


In [ ]:
learn.show_results()


In [ ]:
# marker name to class index
print(dls.vocab)


In [ ]:
import numpy as np

# predictions and targets on the validation set
probs, targets = learn.get_preds(ds_idx=1)

# first window, change idx to look at others
idx = 0  

# one class per window, repeated over its 200 points for plotting
predicted_class = probs[idx].argmax().item()
predicted_zones = np.full(200, predicted_class)

# channel 0 of the window is GR
gr_log = X_valid[idx][0]
seq_len = len(gr_log)

predicted_zones = np.full(seq_len, predicted_class) 
depths = np.arange(seq_len) 


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6), sharey=True)

# left: input GR
ax1.plot(gr_log, depths, color='steelblue', linewidth=1)
ax1.set_title(f'GR Log | Patch {idx}')
ax1.set_ylabel('Depth Steps')
ax1.invert_yaxis()  # depth increases downwards

# right: prediction
ax2.plot(predicted_zones, depths, color='steelblue', drawstyle='steps-pre')
ax2.set_title(f'Model Prediction\n(Guessed: Class {predicted_class} | True: {targets[idx].item()})')
ax2.set_xticks([0, 1, 2, 3, 4, 5]) # one tick per class

plt.tight_layout()
plt.show()


In [ ]:
X_well[:, 1, 100]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# one well, fixed seed
np.random.seed(42)  
well_id = df_valid_tops.index[10]  

X_well, y_well = extract_dataset_Xy(
    df_valid_log, df_valid_tops, [well_id], input_variable, 
    wsize=201, top_list_bool=0
)

probs, targets, _ = learn.get_X_preds(X_well, y_well)
preds = probs.argmax(dim=-1).numpy().flatten()

# depth of each prediction
# channel 1 of X_well is depth, index 100 is the centre of the window
actual_depths = X_well[:, 1, 100]

# sort by depth for the step plot
sort_idx = np.argsort(actual_depths)
actual_depths = actual_depths[sort_idx]
preds = preds[sort_idx]

# full-resolution GR from the dataframe
well_data = df_valid_log[df_valid_log['wellName'] == well_id]
raw_depths = well_data['DEPTH'].values
raw_gr = well_data['GR'].values


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 8), sharey=True)

# left: GR log
ax1.plot(raw_gr, raw_depths, color='#5c85b1', alpha=0.9, linewidth=1)
ax1.set_title(f'GR Log | {well_id}')
ax1.set_xlabel('GR (API)')
ax1.set_ylabel('Depth (ft)')

# depth increases downwards
ax1.invert_yaxis()  

# right: predicted zones
ax2.plot(preds, actual_depths, color='#5c85b1', linewidth=2, drawstyle='steps-pre')
ax2.set_title('Zone step function')
ax2.set_xlabel('Predicted zone')
ax2.set_xticks([0, 1, 2, 3])

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# one well, fixed seed
np.random.seed(42)  
well_id = df_valid_tops.index[0]  

X_well, y_well = extract_dataset_Xy(
    df_valid_log, df_valid_tops, [well_id], input_variable, 
    wsize=201, top_list_bool=0
)

probs, targets, _ = learn.get_X_preds(X_well, y_well)
preds = probs.argmax(dim=-1).numpy().flatten()

# depth of each prediction
# channel 1 of X_well is depth, index 100 is the centre of the window
actual_depths = X_well[:, 1, 100]

# sort by depth for the step plot
sort_idx = np.argsort(actual_depths)
actual_depths = actual_depths[sort_idx]
preds = preds[sort_idx]

# full-resolution GR from the dataframe
well_data = df_valid_log[df_valid_log['wellName'] == well_id]
raw_depths = well_data['DEPTH'].values
raw_gr = well_data['GR'].values


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 8), sharey=True)

# left: GR log
ax1.plot(raw_gr, raw_depths, color='#5c85b1', alpha=0.9, linewidth=1)
ax1.set_title(f'GR Log | {well_id}')
ax1.set_xlabel('GR (API)')
ax1.set_ylabel('Depth (ft)')

# depth increases downwards
ax1.invert_yaxis()  

# right: predicted zones
ax2.plot(preds, actual_depths, color='#5c85b1', linewidth=2, drawstyle='steps-pre')
ax2.set_title('Zone step function')
ax2.set_xlabel('Predicted zone')
ax2.set_xticks([0, 1, 2, 3])

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

formation_names = ['NIOBARA', 'CODELL', 'FORT_HAYS']
top_names = ['zone_0'] + formation_names

# accuracy and classification metrics
probs, targets = learn.get_preds(ds_idx=1)
preds = probs.argmax(dim=-1).numpy().flatten()
targets = targets.numpy().flatten()

print(f"Overall depth-sample accuracy: {(preds == targets).mean():.4f}\n")
print("Per-class metrics:")
print(classification_report(targets, preds, target_names=top_names, digits=2))

print("Confusion matrix (rows=true, cols=pred):")
cm = confusion_matrix(targets, preds)
print(pd.DataFrame(cm, index=top_names, columns=top_names))
print("\n" + "="*50 + "\n")

# top-picking MAE
mae_errors = {name: [] for name in formation_names}
print("Top-picking mean absolute error:")

for well_id in df_valid_tops.index:
    true_tops = df_valid_tops.loc[well_id] 
    
    X_well, y_well = extract_dataset_Xy(
        df_valid_log, df_valid_tops, [well_id], input_variable, 
        wsize=201, top_list_bool=0
    )
    if len(X_well) == 0: continue
    
    depths = X_well[:, 1, 100] 
    probs, _, _ = learn.get_X_preds(X_well, y_well)
    well_preds = probs.argmax(dim=-1).numpy().flatten()
    
    sort_idx = np.argsort(depths)
    depths = depths[sort_idx]
    well_preds = well_preds[sort_idx]

    
    for zone_idx, formation in enumerate(formation_names, start=1):
        true_depth = true_tops[formation]
        pred_idx = np.where(well_preds == zone_idx)[0]
        
        if len(pred_idx) > 0 and not pd.isna(true_depth):
            predicted_depth = depths[pred_idx[0]]
            mae_errors[formation].append(abs(predicted_depth - true_depth))

for formation in formation_names:
    if len(mae_errors[formation]) > 0:
        print(f"  {formation:<12} {np.mean(mae_errors[formation]):>5.1f} ft")


In [ ]:
print("Accuracy per marker (True Positive Rate):")
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

for name, acc in zip(top_names, per_class_accuracy):
    print(f"  {name:<12} {acc*100:>5.1f}%")


In [ ]:
print("Accuracy per marker (True Positive Rate):")
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

for name, acc in zip(top_names, per_class_accuracy):
    print(f"  {name:<12} {acc*100:>5.1f}%")